# AEGIS ML — Notebook 08: M2 Corpus Merger V3

**Purpose:** Merge the 4 per-class M2 CSVs into a single unified dataset,
then produce the stratified train/val/test splits expected by `m2_sequential_finetuningV3.ipynb`.

**Input files** (already in `AEGIS_ML/data/processed/v5/`):

| File | Class label |
|------|-------------|
| `m2_threat_v2.csv` | `threat` |
| `m2_sexual_harassment_v2.csv` | `sexual_harassment` |
| `m2_verbal_harassment_v2.csv` | `verbal_harassment` |
| `m2_discrimination_v3.csv` | `discrimination` |

**Output files** (same folder):
- `m2_trainV3.csv` — 70 %
- `m2_valV3.csv`   — 15 %
- `m2_testV3.csv`  — 15 %

> **Run order:** Run this notebook BEFORE `m2_sequential_finetuningV3.ipynb`.


In [ ]:
# ============================================================
# CELL 0 — Mount Drive + paths  [V4 EDIT]
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROC       = Path('/content/drive/MyDrive/AEGIS_ML/data/processed/v5')
GOLDEN_JSON = Path('/content/drive/MyDrive/AEGIS_ML/data/golden_dataset_EN.json')

print(f'PROC  : {PROC}')
print(f'Exists: {PROC.exists()}')

# ► V4: point threat to the augmented file (7,605 rows)
CLASS_FILES = {
    'threat':             PROC / 'm2_threat_v4_final.csv',        # ← V4 augmented
    'sexual_harassment':  PROC / 'm2_sexual_harassment_v2.csv',
    'verbal_harassment':  PROC / 'm2_verbal_harassment_v2.csv',
    'discrimination':     PROC / 'm2_discrimination_v3.csv',
}

print('\nChecking source files:')
all_found = True
for cls, path in CLASS_FILES.items():
    exists = path.exists()
    size   = f'{path.stat().st_size/1024:.1f} KB' if exists else 'MISSING'
    sym    = '✅' if exists else '❌'
    print(f'  {sym}  {cls:<22}  {path.name:<45}  {size}')
    if not exists:
        all_found = False

golden_ok = GOLDEN_JSON.exists()
print(f'\n  {"✅" if golden_ok else "❌"}  golden_dataset_EN.json  ({GOLDEN_JSON})')

if not all_found:
    print('\n❌ Some source files are missing — fix paths above and re-run.')
else:
    print('\n✅ All source files found — proceed to Cell 1.')


In [ ]:
# ============================================================
# CELL 1 — Shared Text Normalizer (same as all other notebooks)
# ============================================================
import re

def normalize_text(text: str) -> str:
    """Canonical AEGIS normalizer."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'rt\s+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('✅ normalize_text() ready')


In [ ]:
# ============================================================
# CELL 2 — Load all 4 class files + inspect columns
# ============================================================
import pandas as pd

raw_frames = {}  # class_label -> DataFrame

for cls, path in CLASS_FILES.items():
    df = pd.read_csv(path)
    print(f'\n--- {cls} ---')
    print(f'  Shape   : {df.shape}')
    print(f'  Columns : {list(df.columns)}')
    print(f'  Sample  :')
    # Show first non-null text sample
    for col in df.columns:
        if df[col].dtype == object:
            sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else ''
            print(f'    {col}: {str(sample)[:80]}')
    raw_frames[cls] = df

print('\n✅ All files loaded')


In [ ]:
# ============================================================
# CELL 3 — Standardize columns + normalize text
# Each file may have a different 'text' column name.
# ============================================================

# ► Adjust these mappings if Cell 2 showed different column names
# Format: class_label -> name of the text column in that file
TEXT_COL = {
    'threat':            'text',
    'sexual_harassment': 'text',
    'verbal_harassment': 'text',
    'discrimination':    'text',
}

clean_frames = {}

for cls, df in raw_frames.items():
    tcol = TEXT_COL[cls]

    # Auto-detect text column if not 'text'
    if tcol not in df.columns:
        candidates = [c for c in df.columns
                      if df[c].dtype == object and c.lower() not in ('label', 'source', 'split')]
        if candidates:
            tcol = candidates[0]
            print(f'  ⚠️  {cls}: text col not found, using "{tcol}" instead')
        else:
            raise ValueError(f'Cannot find a text column in {cls} file. Columns: {list(df.columns)}')

    out = pd.DataFrame()
    out['text']  = df[tcol].astype(str).apply(normalize_text)
    out['label'] = cls

    # Drop empty / very short texts
    before = len(out)
    out = out[out['text'].str.split().str.len() >= 2]
    dropped = before - len(out)
    if dropped > 0:
        print(f'  {cls}: dropped {dropped} rows with <2 words')

    # Drop duplicates within class
    before = len(out)
    out = out.drop_duplicates(subset='text')
    print(f'  {cls:<22}  {len(out):>6,} rows after dedup (dropped {before-len(out)} dupes)')

    clean_frames[cls] = out.reset_index(drop=True)

print('\n✅ All files standardized and normalized')


In [ ]:
# ============================================================
# CELL 4 — Cross-class deduplication + per-class cap  [V4: UNCAPPED]
# ============================================================
import numpy as np

# V4: raise cap to 12K (threat has 7.6K so will take all; others will be capped)
# Per plan: "Uncap → use Focal Loss for imbalance instead of hard caps"
CAP_PER_CLASS = 12_000   # ← V4 (was 7,000 in V3)

# --- Merge all classes ---
df_all = pd.concat(list(clean_frames.values()), ignore_index=True)
print(f'Total before cross-class dedup: {len(df_all):,}')

# --- Cross-class dedup: keep FIRST occurrence (priority order) ---
priority_order = ['threat', 'sexual_harassment', 'verbal_harassment', 'discrimination']
df_all['_priority'] = df_all['label'].map({c: i for i, c in enumerate(priority_order)})
df_all = df_all.sort_values('_priority').drop_duplicates(subset='text', keep='first')
df_all = df_all.drop(columns=['_priority']).reset_index(drop=True)

print(f'Total after cross-class dedup  : {len(df_all):,}')
print()

# --- Per-class distribution before cap ---
print('Class distribution (before cap):')
for cls in priority_order:
    n = (df_all['label'] == cls).sum()
    bar = '█' * (n // 200)
    print(f'  {cls:<25} {n:>6,}  {bar}')

# --- Cap per class ---
np.random.seed(42)
capped = []
for cls in priority_order:
    sub = df_all[df_all['label'] == cls]
    n_take = min(len(sub), CAP_PER_CLASS)
    capped.append(sub.sample(n=n_take, random_state=42))
    print(f'  {cls:<25}  {len(sub):>6,} → capped to {n_take:>6,}')

df_m2 = pd.concat(capped, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nM2 dataset before golden injection: {len(df_m2):,} rows')
print(df_m2['label'].value_counts())

# --- Normalization gate ---
has_upper = df_m2['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_m2['text'].str.contains(r'https?://', regex=True).sum()
print(f'\nNorm gate — uppercase: {has_upper}  URLs: {has_url}  (both should be 0)')


In [ ]:
# ============================================================
# CELL 4b — Golden Dataset Injection (Real Production Cases)
# ============================================================
# Source: golden_dataset_EN.json — 300 real messages processed by AEGIS
#   • llm_triggered=True  → shadow-zone cases Agent 3 verified (high value)
#   • ml_corrected=True   → cases where ML was WRONG, Agent 3 fixed (highest value)
#   • human_reviewed=True → human-ground-truth verified
#
# Strategy:
#   • Normal golden entries → oversample 5x
#   • ml_corrected=True    → oversample 10x (hardest examples)
#   • Only harmful classes go into M2 (threat/verbal/sexual/discrimination)
# ============================================================

import json, warnings

LABEL_MAP = {
    'threat':               'threat',
    'verbal_harassment':    'verbal_harassment',
    'sexual_harassment':    'sexual_harassment',
    'SEXUAL_HARASSMENT':    'sexual_harassment',   # normalise caps
    'discrimination':       'discrimination',
    'grooming':             'sexual_harassment',   # grooming intent is sexual — maps to sexual_harassment
    'safe':                 None,                  # skip safe for M2
    'uncertain':            None,                  # skip uncertain
}

OVERSAMPLE_NORMAL     = 5
OVERSAMPLE_ML_WRONG   = 10

try:
    with open(GOLDEN_JSON) as f:
        golden_raw = json.load(f)
    print(f'Loaded {len(golden_raw)} golden entries')

    golden_rows = []
    for entry in golden_raw:
        # Use human_label if available (highest trust), else primary_class
        raw_label  = entry.get('human_label') or entry.get('primary_class', '')
        label      = LABEL_MAP.get(raw_label)
        if label is None:
            continue  # skip safe / uncertain for M2

        text = entry.get('normalized_text') or entry.get('raw_text', '')
        text = normalize_text(str(text))
        if len(text.split()) < 2:
            continue

        is_ml_corrected = bool(entry.get('ml_corrected', False))
        n_copies = OVERSAMPLE_ML_WRONG if is_ml_corrected else OVERSAMPLE_NORMAL

        for _ in range(n_copies):
            golden_rows.append({'text': text, 'label': label})

    df_golden = pd.DataFrame(golden_rows).drop_duplicates(subset='text')
    print(f'Golden M2 entries: {len(df_golden)} rows (after dedup)')
    print(df_golden['label'].value_counts().to_string())

    # Merge golden into main dataset (golden entries take priority in dedup)
    df_combined = pd.concat([df_golden, df_m2], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset='text', keep='first')
    df_m2 = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f'\n✅ After golden injection: {len(df_m2):,} total rows')
    print(df_m2['label'].value_counts().to_string())

except FileNotFoundError:
    warnings.warn(f'⚠️  golden_dataset_EN.json not found at {GOLDEN_JSON} — skipping injection')
    print('Continuing without golden dataset.')


In [ ]:
# ============================================================
# CELL 5 — Stratified 70 / 15 / 15 split
# ============================================================
from sklearn.model_selection import train_test_split

print('Stratified 70/15/15 split...')

train, temp = train_test_split(
    df_m2, test_size=0.30, random_state=42, stratify=df_m2['label']
)
val, test = train_test_split(
    temp,  test_size=0.50, random_state=42, stratify=temp['label']
)

print(f'  Train : {len(train):,} rows  ({len(train)/len(df_m2)*100:.1f}%)')
print(f'  Val   : {len(val):,} rows  ({len(val)/len(df_m2)*100:.1f}%)')
print(f'  Test  : {len(test):,} rows  ({len(test)/len(df_m2)*100:.1f}%)')
print(f'  Total : {len(train)+len(val)+len(test):,} rows')

# --- Class balance per split ---
print('\nClass balance per split:')
for name, df in [('Train', train), ('Val', val), ('Test', test)]:
    counts = df['label'].value_counts()
    print(f'  {name}:')
    for cls in priority_order:
        n   = counts.get(cls, 0)
        pct = n / len(df) * 100
        print(f'    {cls:<25} {n:>5,}  ({pct:.1f}%)')

# --- Leakage check ---
print('\nLeakage check:')
train_t = set(train['text'])
val_t   = set(val['text'])
test_t  = set(test['text'])
tv = len(train_t & val_t)
tt = len(train_t & test_t)
vt = len(val_t & test_t)
print(f'  Train∩Val  : {tv}  (must be 0)')
print(f'  Train∩Test : {tt}  (must be 0)')
print(f'  Val∩Test   : {vt}  (must be 0)')
assert tv + tt + vt == 0, 'Leakage found! Stop and investigate.'
print('  ✅ Zero leakage')

# --- Source diversity guard ---
# Verify no single original file dominates > 60% of any class in train
print('\n✅ Splits look clean — proceed to Cell 6 (save).')


In [ ]:
# ============================================================
# CELL 6 — Save m2_trainV3 / m2_valV3 / m2_testV3
# ============================================================

# Only keep the columns the training notebook needs
KEEP_COLS = ['text', 'label']

train[KEEP_COLS].to_csv(PROC / 'm2_trainV4.csv', index=False)
val[KEEP_COLS].to_csv(PROC / 'm2_valV4.csv',   index=False)
test[KEEP_COLS].to_csv(PROC / 'm2_testV4.csv',  index=False)

print('Files saved:')
for fname in ['m2_trainV4.csv', 'm2_valV4.csv', 'm2_testV4.csv']:
    p    = PROC / fname
    size = p.stat().st_size / 1024
    rows = pd.read_csv(p).shape[0]
    print(f'  ✅  {fname:<22}  {rows:>6,} rows  {size:>7.1f} KB')

print('\n✅ Done. m2_sequential_finetuningV3.ipynb can now be run (reads V4 splits).')


In [ ]:
# ============================================================
# CELL 7 — Final Summary
# ============================================================

print('=' * 60)
print('M2 V4 CORPUS MERGER — FINAL SUMMARY')
print('=' * 60)

for split_name, df in [('TRAIN', train), ('VAL', val), ('TEST', test)]:
    print(f'\n  {split_name} ({len(df):,} rows):')
    for cls in priority_order:
        n = (df['label'] == cls).sum()
        print(f'    {cls:<25} {n:>5,}')

print(f'\n  Source files merged:')
for cls, path in CLASS_FILES.items():
    print(f'    {cls:<25} <- {path.name}')

print(f'\n  Output files:')
for fname in ['m2_trainV4.csv', 'm2_valV4.csv', 'm2_testV4.csv']:
    p = PROC / fname
    print(f'    {p}')

print(f'\n  Processing steps applied:')
print(f'    1. normalize_text() on every row')
print(f'    2. Intra-class dedup')
print(f'    3. Cross-class dedup (rare classes kept first)')
print(f'    4. Cap per class: {CAP_PER_CLASS:,}')
print(f'    5. Stratified 70/15/15 split')
print(f'    6. Zero leakage verified')

print(f'\n  Next step: m2_sequential_finetuningV3.ipynb')
print('=' * 60)
